# Round 3: 5-fold OOF checkpoint - S6E10

This round is NOT a new attempt to squeeze the score - it is the checkpoint the log has been
promising. Every number so far (Exp 0-7) comes from one stratified 80/20 holdout, and that same
holdout has now guided tree counts, feature sets, and blend choices. Its estimates drift optimistic,
so before any more tuning we re-score the top configs under 5-fold stratified out-of-fold (OOF)
validation on all 699,635 training rows - an ALTERNATIVE PARTITIONING of the same data (the folds
overlap the old holdout; this is not untouched data).

What this gives us and what it does not:
- Per config: 5 fold scores and a mean +/- spread - a sturdier, more descriptive estimate than one holdout.
- Descriptive evidence on the open questions: does the engineered feature set help LightGBM? Does blending help?
- It does NOT produce a submission file this round. The incumbent (Exp 2 raw LightGBM) stays the best
  submission candidate until a better option is supported beyond this checkpoint.

Reading rules going in (so the result interprets itself):
- Fold scores share most of their training data, so they are correlated, and this data has been selected
  on repeatedly. A mean delta that clears the fold-to-fold spread is a WORKING SIGNAL, not proof.
- The blend weight scan is exploratory: the weight is chosen on the same OOF predictions it is scored on.
- If a delta is within the fold spread: NO SIGNAL - no action.
- 0.96 remains the local target; an OOF mean >= 0.96 would be encouraging, but any target claim stays
  cautious because of the repeated selection above.


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
N_SPLITS = 5
INCUMBENT_HOLDOUT = 0.95760  # Exp 2 raw LightGBM on the old holdout, for agreement check only
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])

def to_lgbm(df):
    # LightGBM-native copy: categoricals as 'category', NaN left alone. sklearn frames keep object dtype.
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

RATING_COLUMNS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness',
]

def add_features(df):
    out = df.copy()
    dep = out['Departure Delay in Minutes']
    arr = out['Arrival Delay in Minutes']
    out['arr_delay_missing'] = arr.isna().astype(int)
    out['total_delay'] = dep + arr
    out['delay_diff'] = arr - dep
    out['dep_delayed'] = (dep > 0).astype(int)
    out['arr_delayed'] = (arr > 0).astype(int)
    out['service_mean'] = out[RATING_COLUMNS].mean(axis=1)
    out['service_min'] = out[RATING_COLUMNS].min(axis=1)
    out['service_std'] = out[RATING_COLUMNS].std(axis=1)
    out['low_service_count'] = (out[RATING_COLUMNS] <= 2).sum(axis=1)
    out['log_flight_distance'] = np.log1p(out['Flight Distance'])
    return out

def rf_pipeline(train_frame, max_features='sqrt', min_samples_leaf=1, n_estimators=200):
    numeric_columns = train_frame.columns.difference(CATEGORICAL).tolist()
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
            max_features=max_features, n_jobs=-1, random_state=RANDOM_STATE)),
    ])

features_fe = add_features(features)
print(f'Full train: {len(features):,} rows, {features.shape[1]} raw / {features_fe.shape[1]} engineered features')


Full train: 699,635 rows, 21 raw / 31 engineered features


## Protocol

Three configs go through the same 5 folds (stratified, shuffled, seed 42):

- **A: incumbent** - raw LightGBM (Exp 2 config: lr=0.05, num_leaves=63, early stop 50).
- **B: LightGBM + engineered set** - Exp 6c config, the 10-feature set.
- **C: random forest** - Exp 4 config (400 trees, min_samples_leaf=5), raw features.

Early stopping for the LightGBM runs uses an INNER 10% split of each fold's training part -
never the fold being evaluated - so tree-count selection can't leak into the fold scores. The random
forest needs no early stopping.

Honesty notes for the log: fold scores are correlated (shared training rows), and the blend weight is
picked on the same OOF it is scored on - everything here is descriptive evidence, and the weight scan
is exploratory. Deltas are called WORKING SIGNAL / NO SIGNAL accordingly, never confirmed effects.

Expected runtime: ~20-30 min total (the 5 random-forest fits dominate; LightGBM folds are fast).


In [3]:
def oof_lgbm(feat_frame, label, name):
    frame = to_lgbm(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        model = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        p = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p
        a = roc_auc_score(y_va, p)
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

print('Config A: incumbent raw LightGBM')
oof_a, aucs_a, iters_a = oof_lgbm(features, target, 'A')
print(f'  A mean OOF AUC: {np.mean(aucs_a):.5f} +/- {np.std(aucs_a):.5f}')

print('Config B: LightGBM + engineered set')
oof_b, aucs_b, iters_b = oof_lgbm(features_fe, target, 'B')
print(f'  B mean OOF AUC: {np.mean(aucs_b):.5f} +/- {np.std(aucs_b):.5f}')


Config A: incumbent raw LightGBM


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  A fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  A fold 2: AUC 0.95751 (best_iteration=265)


...

  B fold 3: AUC 0.95896 (best_iteration=182)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  B fold 4: AUC 0.95837 (best_iteration=243)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  B fold 5: AUC 0.95865 (best_iteration=215)
  B mean OOF AUC: 0.95831 +/- 0.00056


In [4]:
print('Config C: random forest 400 trees, leaf=5 (raw features)')
oof_c = np.zeros(len(features))
aucs_c = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(features, target), 1):
    X_tr, y_tr = features.iloc[tr_idx], target.iloc[tr_idx]
    X_va, y_va = features.iloc[va_idx], target.iloc[va_idx]
    rf = rf_pipeline(X_tr, n_estimators=400, min_samples_leaf=5)
    rf.fit(X_tr, y_tr)
    p = rf.predict_proba(X_va)[:, 1]
    oof_c[va_idx] = p
    a = roc_auc_score(y_va, p)
    aucs_c.append(a)
    print(f'  C fold {fold}: AUC {a:.5f}')
print(f'  C mean OOF AUC: {np.mean(aucs_c):.5f} +/- {np.std(aucs_c):.5f}')


Config C: random forest 400 trees, leaf=5 (raw features)


  C fold 1: AUC 0.95664


  C fold 2: AUC 0.95579


  C fold 3: AUC 0.95697


  C fold 4: AUC 0.95688


  C fold 5: AUC 0.95662
  C mean OOF AUC: 0.95658 +/- 0.00042


## Blend on OOF

The round-2 blend used a fixed 50/50 weight picked on the holdout. Here the weight is chosen on OOF
predictions - better than the holdout version, but still EXPLORATORY: the weight is selected on the
same OOF predictions it is then scored on, so its AUC is optimistically biased.


In [5]:
blend_rows = []
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = w * oof_a + (1 - w) * oof_c
    blend_rows.append((f'blend {w:.1f}*A + {1-w:.1f}*C', roc_auc_score(target, p)))
blend_table = pd.DataFrame(blend_rows, columns=['blend', 'oof_auc']).sort_values('oof_auc', ascending=False)
print(blend_table.to_string(index=False))
best_blend_name, best_blend_auc = blend_rows[np.argmax([r[1] for r in blend_rows])]


              blend  oof_auc
blend 0.7*A + 0.3*C 0.958322
blend 0.6*A + 0.4*C 0.958118
blend 0.5*A + 0.5*C 0.957884
blend 0.4*A + 0.6*C 0.957636
blend 0.3*A + 0.7*C 0.957378


## Results and verdicts

The summary below is the log-worthy output. Labels follow the reading rules at the top: WORKING SIGNAL
when a mean delta clears the fold-to-fold spread (descriptive, not proof - correlated folds, repeated
selection), NO SIGNAL otherwise. Nothing here is a confirmed effect.


In [6]:
def verdict(delta, spread):
    if abs(delta) <= 2 * spread:
        return 'NO SIGNAL (within fold-to-fold spread - no action)'
    return ('WORKING SIGNAL: better' if delta > 0 else 'WORKING SIGNAL: worse') + ' (clears 2x fold spread; descriptive, not proof - correlated folds, repeated selection)'

incumbent_oof = np.mean(aucs_a)
rows = [
    ('A incumbent raw LGBM', aucs_a, np.mean(aucs_a), np.std(aucs_a), np.nan, '-'),
    ('B LGBM + engineered', aucs_b, np.mean(aucs_b), np.std(aucs_b), np.mean(aucs_b) - incumbent_oof,
     verdict(np.mean(aucs_b) - incumbent_oof, np.std([b - a for a, b in zip(aucs_a, aucs_b)]))),
    ('C RF 400/leaf5', aucs_c, np.mean(aucs_c), np.std(aucs_c), np.mean(aucs_c) - incumbent_oof,
     verdict(np.mean(aucs_c) - incumbent_oof, np.std([c - a for a, c in zip(aucs_a, aucs_c)]))),
    (best_blend_name, None, best_blend_auc, np.nan, best_blend_auc - incumbent_oof,
     verdict(best_blend_auc - incumbent_oof, np.std(aucs_a))),
]
summary = pd.DataFrame(rows, columns=['config', 'fold_aucs', 'oof_mean', 'oof_std', 'delta_vs_incumbent', 'verdict'])
summary['fold_aucs'] = summary['fold_aucs'].apply(lambda v: '-' if v is None else ' '.join(f'{a:.5f}' for a in v))
print(summary.to_string(index=False))

print()
print(f'Incumbent agreement check: holdout said {INCUMBENT_HOLDOUT:.5f}, OOF mean says {incumbent_oof:.5f} '
      f'(gap {incumbent_oof - INCUMBENT_HOLDOUT:+.5f} - a negative gap is the expected holdout optimism).')
best_oof = max(incumbent_oof, np.mean(aucs_b), np.mean(aucs_c), best_blend_auc)
print(f'Best OOF mean: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

print()
print('Decisions per the rules:')
if abs(np.mean(aucs_b) - incumbent_oof) <= 2 * np.std([b - a for a, b in zip(aucs_a, aucs_b)]):
    print('- Feature set on LightGBM: NO SIGNAL -> set aside for trees (stays on the table for logistic).')
elif np.mean(aucs_b) < incumbent_oof:
    print('- Feature set on LightGBM: working signal of harm -> dropped for trees.')
else:
    print('- Feature set on LightGBM: working signal of improvement -> candidate default; re-check before adopting.')
if best_blend_auc > incumbent_oof and abs(best_blend_auc - incumbent_oof) > 2 * np.std(aucs_a):
    print(f'- Blend: working signal -> {best_blend_name} refit becomes a submission CANDIDATE; remember the weight was chosen on this same OOF (exploratory).')
else:
    print('- Blend: no signal over the incumbent -> incumbent stands; no new submission candidate.')
print('- No submission file written this round; checkpoint only.')


              config                               fold_aucs  oof_mean  oof_std  delta_vs_incumbent                                              verdict
A incumbent raw LGBM 0.95851 0.95751 0.95907 0.95841 0.95885  0.958470 0.000534                 NaN                                                    -
 B LGBM + engineered 0.95825 0.95731 0.95896 0.95837 0.95865  0.958308 0.000558           -0.000162                   WORKING SIGNAL: worse (clears 2x fold spread; descriptive, not proof)
      C RF 400/leaf5 0.95664 0.95579 0.95697 0.95688 0.95662  0.956582 0.000420           -0.001888                   WORKING SIGNAL: worse (clears 2x fold spread; descriptive, not proof)
 blend 0.7*A + 0.3*C                                       -  0.958322      NaN           -0.000148 NO SIGNAL (within fold-to-fold spread - no action)

Incumbent agreement check: holdout said 0.95760, OOF mean says 0.95847 (gap +0.00087 - a negative gap is the expected holdout optimism).
Best OOF mean reached >=0.96 

## What happens next

Depending on the signals above, exactly one of these is round 4 (one idea, as always):

- num_leaves / learning_rate variation on the OOF leader, one knob at a time, selected on OOF -
  with the same descriptive framing; repeated selection on one dataset has a real multiple-comparisons cost.
- If the blend showed a working signal: refit both members on all rows and write a blend submission
  file (still Andrea's call to submit from his own account), logged as exploratory.
- If nothing shows a signal and OOF sits clearly below the target: the honest options are patient
  one-knob tuning under OOF, or the original passenger-survey dataset as extra training data -
  that one needs Andrea's explicit OK first (provenance/licensing check).

Round-3 runtime and any execution fixes belong in the log's Run notes, as usual.
